In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

In [1]:
##############################################################################
# 0. Reproducibility / Device Config
##############################################################################
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

##############################################################################
# 1. Chunked Dataset for BERT
##############################################################################
class ChunkedTextDataset(Dataset):
    """
    Splits each document into multiple overlapping chunks if it exceeds `max_chunk_len` tokens.
    Each chunk is treated as a separate sample, labeled the same as the full document.
    """
    def __init__(self, texts, labels, tokenizer, max_chunk_len=256, overlap=50):
        self.tokenizer = tokenizer
        self.max_chunk_len = max_chunk_len
        self.overlap = overlap

        self.chunks = []
        self.labels = []

        for text, label in zip(texts, labels):
            # Convert text to token IDs
            token_ids = tokenizer.encode(text, add_special_tokens=True)
            start = 0

            while True:
                end = start + max_chunk_len
                chunk = token_ids[start:end]
                self.chunks.append(chunk)
                self.labels.append(label)

                if end >= len(token_ids):
                    # No more chunks to take
                    break
                # Overlap handling
                start = end - overlap
                if start < 0:
                    break

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        token_ids = self.chunks[idx]
        label = self.labels[idx]

        # Pad/truncate to max_chunk_len (should not exceed due to chunking, but just in case)
        pad_length = self.max_chunk_len - len(token_ids)
        if pad_length < 0:
            token_ids = token_ids[:self.max_chunk_len]
            pad_length = 0

        attention_mask = [1] * len(token_ids) + [0] * pad_length
        token_ids += [tokenizer.pad_token_id] * pad_length

        return {
            'input_ids': torch.tensor(token_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

##############################################################################
# 2. BERT + Feed-Forward Head
##############################################################################
class BertFeedForwardClassifier(nn.Module):
    def __init__(self, bert_model_name, num_labels, hidden_size=256, dropout_rate=0.1):
        super(BertFeedForwardClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.ffn1 = nn.Linear(self.bert.config.hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        self.ffn2 = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        x = self.dropout1(cls_embedding)
        x = self.ffn1(x)
        x = self.relu(x)
        x = self.dropout2(x)
        logits = self.ffn2(x)
        return logits

##############################################################################
# 3. Load Data
##############################################################################
df = pd.read_csv("nace_codes.csv", encoding="ISO-8859-1")  # Update to your file if needed
X_all = df["report_text"].tolist()  # Column with text
y_all = df["nace_code"].tolist()    # Column with NACE code labels

label_encoder = LabelEncoder()
y_all_encoded = label_encoder.fit_transform(y_all)
num_labels = len(label_encoder.classes_)
print(f"Number of unique NACE classes: {num_labels}")

##############################################################################
# 4. Split Data: Train / Cal / Test
##############################################################################
#  - Train set: used to train the model
#  - Cal (Calibration) set: used to compute nonconformities for Conformal Prediction
#  - Test set: final evaluation and coverage measurement
X_train_cal, X_test, y_train_cal, y_test = train_test_split(
    X_all, y_all_encoded,
    test_size=0.15,  # 15% test
    stratify=y_all_encoded,
    random_state=SEED
)

X_train, X_cal, y_train, y_cal = train_test_split(
    X_train_cal, y_train_cal,
    test_size=0.1765,  # ~ (15 / 85) => total 70% train, 15% cal, 15% test
    stratify=y_train_cal,
    random_state=SEED
)

print("Train size:", len(X_train), "Cal size:", len(X_cal), "Test size:", len(X_test))

##############################################################################
# 5. Create Chunked Datasets & DataLoaders
##############################################################################
bert_model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(bert_model_name)

MAX_CHUNK_LEN = 256
OVERLAP = 50
BATCH_SIZE = 8
EPOCHS = 30

train_dataset = ChunkedTextDataset(X_train, y_train, tokenizer, max_chunk_len=MAX_CHUNK_LEN, overlap=OVERLAP)
cal_dataset   = ChunkedTextDataset(X_cal,   y_cal,   tokenizer, max_chunk_len=MAX_CHUNK_LEN, overlap=OVERLAP)
test_dataset  = ChunkedTextDataset(X_test,  y_test,  tokenizer, max_chunk_len=MAX_CHUNK_LEN, overlap=OVERLAP)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
cal_loader   = DataLoader(cal_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Chunked train samples:", len(train_dataset))
print("Chunked cal samples:", len(cal_dataset))
print("Chunked test samples:", len(test_dataset))

##############################################################################
# 6. Class Weights (Optional, for Imbalance)
##############################################################################
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),  # training set classes
    y=y_train
)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Class Weights:", class_weights)

##############################################################################
# 7. Initialize Model, Loss, Optimizer
##############################################################################
model = BertFeedForwardClassifier(
    bert_model_name=bert_model_name,
    num_labels=num_labels,
    hidden_size=256,
    dropout_rate=0.1
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

##############################################################################
# 8. Train Model on TRAIN set
##############################################################################
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}")

# (We do not do early stopping here since we have a separate calibration set for conformal)


/Users/hendrikweichel/miniconda3/envs/remedi/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Users/hendrikweichel/miniconda3/envs/remedi/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <61623A3D-DA3C-3AAD-B2F0-D363151DDB3F> /Users/hendrikweichel/miniconda3/envs/remedi/lib/python3.10/site-packages/torchvision/image.so
  Expected in:     <E3D17B4A-4867-3D49-BC92-E04C28EE0F45> /Users/hendrikweichel/miniconda3/envs/remedi/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib
  warn(f"Failed to load image Python extension: {e}")


Using device: cpu
Number of unique NACE classes: 21
Train size: 688 Cal size: 148 Test size: 148
Chunked train samples: 688
Chunked cal samples: 148
Chunked test samples: 148
Class Weights: [ 0.88545689  1.82010582  0.15238095  1.09206349  1.09206349  1.09206349
  0.42547928  0.60670194  4.68027211  1.05683564  0.93605442  8.19047619
  1.5600907   1.02380952  4.0952381   2.97835498  2.73015873  2.34013605
  1.82010582 16.38095238 16.38095238]


Training Epoch 1/30: 100%|██████████| 86/86 [03:19<00:00,  2.32s/it]


Epoch 1, Train Loss: 2.9150


Training Epoch 2/30: 100%|██████████| 86/86 [02:57<00:00,  2.07s/it]


...

Epoch 28, Train Loss: 0.0149


Training Epoch 29/30: 100%|██████████| 86/86 [02:39<00:00,  1.86s/it]


Epoch 29, Train Loss: 0.0155


Training Epoch 30/30: 100%|██████████| 86/86 [02:32<00:00,  1.78s/it]

Epoch 30, Train Loss: 0.0138


In [ ]:
##############################################################################
# Save model
torch.save(model.state_dict(), "bert_ffn.pt")
##############################################################################

In [3]:
# Load the saved model state
model.load_state_dict(torch.load("bert_ffn.pt", map_location=device))
model.to(device)
model.eval()  # Set the model to evaluation mode
print("Model loaded successfully.")

Model loaded successfully.


In [6]:
##############################################################################
# 9. Compute Nonconformities on CAL set (for Conformal Prediction)
##############################################################################
model.eval()
nonconformities = []

with torch.no_grad():
    for batch in cal_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        probs = nn.Softmax(dim=1)(logits)

        # For each sample in the batch
        for i in range(len(labels)):
            true_label = labels[i].item()
            p_correct = probs[i, true_label].item()
            # nonconformity = 1 - probability_of_true_label
            nonconf = 1.0 - p_correct
            nonconformities.append(nonconf)

nonconformities = np.array(nonconformities)
print(f"Collected {len(nonconformities)} nonconformities from calibration set.")

# Choose alpha => coverage ~ (1 - alpha)
alpha = 0.12  # e.g., 90% target coverage
tau = np.quantile(nonconformities, 1 - alpha)
print(f"Conformal threshold (tau) for alpha={alpha}: {tau:.4f}")

##############################################################################
# 10. Evaluate Coverage on TEST set
##############################################################################
correct_coverage = 0
total_samples = 0
prediction_set_sizes = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        probs = nn.Softmax(dim=1)(logits).cpu().numpy()

        for i in range(len(labels)):
            true_label = labels[i].item()
            p_vec = probs[i]
            # Build conformal set => all classes whose (1 - p) <= tau
            pred_set = []
            for class_idx, p_val in enumerate(p_vec):
                if (1.0 - p_val) <= tau:
                    pred_set.append(class_idx)

            prediction_set_sizes.append(len(pred_set))
            if true_label in pred_set:
                correct_coverage += 1
            total_samples += 1

coverage_rate = correct_coverage / total_samples
avg_set_size = np.mean(prediction_set_sizes)

print(f"\nCoverage on Test Set: {coverage_rate:.3f} (Expected ~ {1 - alpha})")
print(f"Average Prediction Set Size: {avg_set_size:.2f}")

##############################################################################
# 11. Example Conformal Prediction on New Text
##############################################################################
def conformal_predict(model, text, tau) :
    """
    Produces a set of class indices whose nonconformities (1 - prob) are <= tau.
    Because we chunk texts at inference, we'll do a simple approach:
      - Tokenize and chunk
      - Evaluate each chunk
      - Aggregate (e.g., union) the sets across all chunks
    """
    model.eval()

    token_ids = tokenizer.encode(text, add_special_tokens=True)
    chunk_size = MAX_CHUNK_LEN
    overlap = OVERLAP

    all_pred_sets = []
    start = 0
    while True:
        end = start + chunk_size
        chunk = token_ids[start:end]

        # Pad if needed
        pad_length = chunk_size - len(chunk)
        attn_mask = [1]*len(chunk) + [0]*pad_length
        chunk += [tokenizer.pad_token_id]*pad_length

        input_ids_t = torch.tensor([chunk], dtype=torch.long).to(device)
        attn_mask_t = torch.tensor([attn_mask], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_ids_t, attn_mask_t)
            probs = nn.Softmax(dim=1)(logits).cpu().numpy()[0]

        pred_set_chunk = []
        for class_idx, p_val in enumerate(probs):
            if (1.0 - p_val) <= tau:
                pred_set_chunk.append(class_idx)
        all_pred_sets.append(set(pred_set_chunk))

        if end >= len(token_ids):
            break
        start = end - overlap
        if start < 0:
            break

    # Final set could be union or intersection. Typically union => if any chunk strongly suggests a class.
    final_set = set()
    for s in all_pred_sets:
        final_set = final_set.union(s)

    return list(final_set)

# Demonstration
new_text = "This year, we focused on growing various cereals and expanding our agricultural capacity."
pred_set_indices = conformal_predict(model, new_text, tau)
pred_set_labels = [label_encoder.inverse_transform([idx])[0] for idx in pred_set_indices]

print("\nNew Text:")
print(new_text)
print("Conformal Prediction Set (class indices):", pred_set_indices)
print("Conformal Prediction Set (labels):", pred_set_labels)

##############################################################################
# 12. Save Model & Encoder
##############################################################################
torch.save(model.state_dict(), "bert_ffn_conformal.pt")
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
print("\nModel + label encoder saved.")
with open("tau.pkl", "wb") as f:
    pickle.dump(tau, f)

Collected 148 nonconformities from calibration set.
Conformal threshold (tau) for alpha=0.12: 0.0420

Coverage on Test Set: 0.878 (Expected ~ 0.88)
Average Prediction Set Size: 0.90

New Text:
This year, we focused on growing various cereals and expanding our agricultural capacity.
Conformal Prediction Set (class indices): [0]
Conformal Prediction Set (labels): ['A']

Model + label encoder saved.


In [ ]:
new_report = "Zalando is selling fashion, footwear, and beauty products from various brands."
predicted_set_indices = conformal_predict(model, new_report, tau)
predicted_labels = [label_encoder.inverse_transform([idx])[0] for idx in predicted_set_indices]

print("\nNew Report:", new_report)
print("Conformal Prediction Set (class indices):", predicted_set_indices)
print("Conformal Prediction Set (labels):", predicted_labels)


New Report: Zalando is selling fashion, footwear, and beauty products from various brands.
Conformal Prediction Set (class indices): [6]
Conformal Prediction Set (labels): ['G']


In [ ]:
def predict_single_label(model, tokenizer, text, device, max_chunk_len=256, overlap=50):
    model.eval()

    token_ids = tokenizer.encode(text, add_special_tokens=True)
    start = 0
    chunk_logits = []

    while True:
        end = start + max_chunk_len
        chunk = token_ids[start:end]

        # Pad if needed
        pad_length = max_chunk_len - len(chunk)
        attn_mask = [1] * len(chunk) + [0] * pad_length
        chunk += [tokenizer.pad_token_id] * pad_length

        # Move to the same device as the model
        input_ids_t = torch.tensor([chunk], dtype=torch.long).to(device)
        attn_mask_t = torch.tensor([attn_mask], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_ids_t, attn_mask_t)  # shape [1, num_labels]
            logits = logits.squeeze(0).cpu().numpy()  # shape [num_labels]

        chunk_logits.append(logits)

        if end >= len(token_ids):
            break
        start = end - overlap
        if start < 0:
            break

    aggregated_logits = np.mean(chunk_logits, axis=0)
    pred_class_idx = np.argmax(aggregated_logits)
    return pred_class_idx


In [ ]:
# After training / loading model + label_encoder:
new_text = "Manufacture of prepared animal feeds"

pred_class_idx = predict_single_label(model, tokenizer, new_text, device=device)
pred_label = label_encoder.inverse_transform([pred_class_idx])[0]

print("Predicted Class Index:", pred_class_idx)
print("Predicted NACE Code:", pred_label)


Predicted Class Index: 0
Predicted NACE Code: A
